#### Simple Gen AI APP Using Langchain

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [2]:
from langchain_community.document_loaders import WebBaseLoader

C:\Users\antoc\AppData\Local\Temp\ipykernel_28576\967698044.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
d:\Udemy_Generative_AI_course\Enterprise_AI_Engineer_RoadMap\04-LangChain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
loader =WebBaseLoader("https://docs.langchain.com/langsmith/observability-llm-tutorial")
loader

In [4]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content='Trace an LLM application tutorial - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentFall AMA Series: Live sessions on building, evaluating, deploying, and continously improving agents with LangSmith. Register now →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialOverviewTraceDebugObserveReferenceQuickstartTutorialConceptsChatTracing setupIntegrationsManual instrumentationConfiguration & troubleshootingProject & environment settingsCost trackingUsage and billingAdva

In [5]:
## Load data divide in to chunks
## convert the chunks with Vector using embeddings
## Store in vector DB 

from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs_chunks = text_splitter.split_documents(docs)

In [6]:
docs_chunks

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content='Trace an LLM application tutorial - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentFall AMA Series: Live sessions on building, evaluating, deploying, and continously improving agents with LangSmith. Register now →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialOverviewTraceDebugObserveReferenceQuickstartTutorialConceptsChatTracing setupIntegrationsManual instrumentationConfiguration & troubleshootingProject & environment settingsCost trackingUsage and billingAdva

In [7]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [8]:
from langchain_community.vectorstores import FAISS

In [9]:
vectorstoredb=FAISS.from_documents(docs_chunks, embeddings)

In [10]:
vectorstoredb

In [11]:

query="What are the Prerequisites needed to use LangSmith?"
results=vectorstoredb.similarity_search(query)
results[0].page_content

'Trace individual LLM calls and full application pipelines.\nCollect and query user feedback.\nLog metadata and use it for filtering and A/B testing.\nUse monitoring dashboards to track production performance.\n\nThe application will retrieve relevant documentation snippets and use them to answer user questions. The retriever is mocked in this tutorial; in a real application you would replace it with a vector search or similar.\n\u200bPrerequisites\nBefore you begin, make sure you have:\n\nA LangSmith account: Sign up or log in at smith.langchain.com.\nA LangSmith API key: Follow the Create an API key guide.\nAn OpenAI API key: Generate this from the OpenAI dashboard.\nLangSmith CLI (optional): Install to inspect traces from the terminal. For instructions, refer to LangSmith CLI.\n\nInstall the required packages:\nPythonTypeScriptpip install langsmith openai\nnpm install langsmith openai\nnpm install -D typescript tsx'

In [12]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")

In [13]:
# Reterival Chain, Document chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


"""
)

document_chain = create_stuff_documents_chain(prompt=prompt, llm=llm)
document_chain



RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': Fal

In [14]:
reteriver = vectorstoredb.as_retriever()
from langchain_classic.chains.retrieval import create_retrieval_chain

reterival_chain = create_retrieval_chain(reteriver, document_chain)

In [15]:
reterival_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001A0EE19CF20>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
            | 

In [16]:
query

'What are the Prerequisites needed to use LangSmith?'

In [17]:
response = reterival_chain.invoke({"input":"What are the Prerequisites needed to use LangSmith?"})
response['answer']

'What are the prerequisites needed before beginning with the application described in the context? \n\nBefore starting with the application, you need the following prerequisites:\n\n1. A LangSmith account: You can sign up or log in at smith.langchain.com.\n2. A LangSmith API key: You need to follow the guide to create an API key.\n3. An OpenAI API key: This should be generated from the OpenAI dashboard.\n4. LangSmith CLI (optional): For inspecting traces from the terminal, you can install the LangSmith CLI.\n5. Required packages for Python and TypeScript: You need to install the needed packages using the following commands:\n   - For Python: `pip install langsmith openai`\n   - For TypeScript: `npm install langsmith openai` and `npm install -D typescript tsx`'

In [18]:
response

{'input': 'What are the Prerequisites needed to use LangSmith?',
 'context': [Document(id='8e1ef2ac-5093-46e2-b5fa-cd2dc49ac3ae', metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content='Trace individual LLM calls and full application pipelines.\nCollect and query user feedback.\nLog metadata and use it for filtering and A/B testing.\nUse monitoring dashboards to track production performance.\n\nThe application will retrieve relevant documentation snippets and use them to answer user questions. The retriever is mocked in this tutorial; in a real application you would replace it with a vector search or similar.\n\u200bPrerequisites\nBefore you begin, make sure you have:\n\nA LangSmith account: Sign up or log in at smith.langchain.com.\nA LangSm

In [19]:
response['context']

[Document(id='8e1ef2ac-5093-46e2-b5fa-cd2dc49ac3ae', metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content='Trace individual LLM calls and full application pipelines.\nCollect and query user feedback.\nLog metadata and use it for filtering and A/B testing.\nUse monitoring dashboards to track production performance.\n\nThe application will retrieve relevant documentation snippets and use them to answer user questions. The retriever is mocked in this tutorial; in a real application you would replace it with a vector search or similar.\n\u200bPrerequisites\nBefore you begin, make sure you have:\n\nA LangSmith account: Sign up or log in at smith.langchain.com.\nA LangSmith API key: Follow the Create an API key guide.\nAn OpenAI API key: Generate